In [ ]:
from google.colab import files
uploaded = files.upload()

Saving qpu_bruta_lote_07.txt to qpu_bruta_lote_07.txt
Saving qpu_bruta_lote_06.txt to qpu_bruta_lote_06.txt
Saving qpu_bruta_lote_05.txt to qpu_bruta_lote_05.txt
Saving qpu_bruta_lote_04.txt to qpu_bruta_lote_04.txt
Saving qpu_bruta_lote_03.txt to qpu_bruta_lote_03.txt
Saving qpu_bruta_lote_02.txt to qpu_bruta_lote_02.txt
Saving qpu_bruta_lote_01.txt to qpu_bruta_lote_01.txt
Saving qpu_bruta_lote_30.txt to qpu_bruta_lote_30.txt
Saving qpu_bruta_lote_29.txt to qpu_bruta_lote_29.txt
Saving qpu_bruta_lote_28.txt to qpu_bruta_lote_28.txt
Saving qpu_bruta_lote_27.txt to qpu_bruta_lote_27.txt
Saving qpu_bruta_lote_26.txt to qpu_bruta_lote_26.txt
Saving qpu_bruta_lote_25.txt to qpu_bruta_lote_25.txt
Saving qpu_bruta_lote_24.txt to qpu_bruta_lote_24.txt
Saving qpu_bruta_lote_23.txt to qpu_bruta_lote_23.txt
Saving qpu_bruta_lote_22.txt to qpu_bruta_lote_22.txt
Saving qpu_bruta_lote_21.txt to qpu_bruta_lote_21.txt
Saving qpu_bruta_lote_20.txt to qpu_bruta_lote_20.txt
Saving qpu_bruta_lote_19.txt

In [ ]:
from pathlib import Path
import math
import hashlib
import sys

import numpy as np
import pandas as pd
import scipy
import statsmodels

from statsmodels.sandbox.stats.runs import runstest_1samp

JOB_ID = "da137vu3kjvs73868gp0"
BACKEND = "ibm_marrakesh"
QUBIT_FISICO = 0

# Caminhos no Google Colab
PASTA_DADOS = Path("/content/dados/bruto")
PASTA_RESULTADOS = Path("/content/results")

PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Job original:", JOB_ID)
print("Backend:", BACKEND)
print("Qubit físico:", QUBIT_FISICO)
print("Pasta de dados:", PASTA_DADOS)

Job original: da137vu3kjvs73868gp0
Backend: ibm_marrakesh
Qubit físico: 0
Pasta de dados: /content/dados/bruto


In [ ]:
from pathlib import Path
import shutil

destino = Path("/content/dados/bruto")
destino.mkdir(parents=True, exist_ok=True)

for nome in uploaded.keys():
    shutil.move(nome, destino / nome)

print("Arquivos movidos:", len(list(destino.glob("qpu_bruta_lote_*.txt"))))

Arquivos movidos: 30


In [ ]:
# Carregamento e validação dos dados brutos
arquivos = sorted(PASTA_DADOS.glob("qpu_bruta_lote_*.txt"))

if len(arquivos) != 30:
    raise ValueError(f"Esperados 30 lotes da QPU, mas foram encontrados {len(arquivos)} em {PASTA_DADOS.resolve()}.")

lotes = []
for arquivo in arquivos:
    bits = arquivo.read_text(encoding="utf-8").strip()
    if len(bits) != 10_000:
        raise ValueError(f"{arquivo.name}: esperado 10.000 bits, encontrado {len(bits)}.")
    if set(bits) - {"0", "1"}:
        raise ValueError(f"{arquivo.name}: conteúdo não binário.")
    lotes.append(bits)

print("Lotes carregados:", len(lotes))
print("Bits por lote:", len(lotes[0]))
print("Total de bits:", sum(len(x) for x in lotes))


Lotes carregados: 30
Bits por lote: 10000
Total de bits: 300000


In [ ]:
# Funções de análise estatística
ALFA = 0.05

def numero_corridas(vetor):
    if len(vetor) == 0:
        return 0
    return int(1 + np.sum(vetor[1:] != vetor[:-1]))

def metricas(bits):
    vetor = np.fromiter((int(b) for b in bits), dtype=np.int8)
    n = len(vetor)
    n1 = int(vetor.sum())
    n0 = n - n1
    p1 = n1 / n
    p0 = n0 / n
    vies = abs(p1 - 0.5)

    probs = [p for p in (p0, p1) if p > 0]
    shannon = -sum(p * math.log2(p) for p in probs)
    min_entropia = -math.log2(max(p0, p1))

    if n > 1 and np.std(vetor[:-1]) > 0 and np.std(vetor[1:]) > 0:
        autocorr = float(np.corrcoef(vetor[:-1], vetor[1:])[0, 1])
    else:
        autocorr = np.nan

    runs_z, runs_p = runstest_1samp(vetor, cutoff=0.5)

    return {
        "n_bits": n,
        "n0": n0,
        "n1": n1,
        "p0": p0,
        "p1": p1,
        "vies": vies,
        "shannon": shannon,
        "min_entropia": min_entropia,
        "autocorr_lag1": autocorr,
        "numero_corridas": numero_corridas(vetor),
        "runs_z": float(runs_z),
        "runs_p": float(runs_p),
        "runs_nao_rejeita_h0_alfa_005": bool(runs_p >= ALFA),
    }


In [ ]:
# Métricas por lote
resultados = []
for i, bits in enumerate(lotes, start=1):
    linha = {
        "lote": i,
        "fonte": "QPU_IBM",
        "job_id": JOB_ID,
        "backend": BACKEND,
        "qubit_fisico": QUBIT_FISICO,
    }
    linha.update(metricas(bits))
    resultados.append(linha)

df_metricas = pd.DataFrame(resultados)
arquivo_metricas = PASTA_RESULTADOS / "qpu_reanalise_metricas.csv"
df_metricas.to_csv(arquivo_metricas, index=False)

print("Arquivo salvo em:", arquivo_metricas)
df_metricas


Arquivo salvo em: /content/results/qpu_reanalise_metricas.csv


,lote,fonte,job_id,backend,qubit_fisico,n_bits,n0,n1,p0,p1,vies,shannon,min_entropia,autocorr_lag1,numero_corridas,runs_z,runs_p,runs_nao_rejeita_h0_alfa_005
0,1,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,4922,5078,0.4922,0.5078,0.0078,0.999824,0.977668,-0.012948,5064,1.284713,0.198893,True
1,2,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,5047,4953,0.5047,0.4953,0.0047,0.999936,0.986502,0.011614,4942,-1.171326,0.241468,True
2,3,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,5038,4962,0.5038,0.4962,0.0038,0.999958,0.989077,-0.026762,5134,2.666063,0.007675,False
3,4,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,5066,4934,0.5066,0.4934,0.0066,0.999874,0.981081,0.015530,4922,-1.562926,0.118070,True
4,5,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,4893,5107,0.4893,0.5107,0.0107,0.999670,0.969452,-0.004761,5022,0.466033,0.641192,True
5,6,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,4983,5017,0.4983,0.5017,0.0017,0.999992,0.995103,-0.006712,5034,0.661197,0.508486,True
6,7,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,4977,5023,0.4977,0.5023,0.0023,0.999985,0.993379,-0.009723,5049,0.962184,0.335957,True
7,8,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,4883,5117,0.4883,0.5117,0.0117,0.999605,0.966630,0.009363,4951,-0.945809,0.344246,True
8,9,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,4962,5038,0.4962,0.5038,0.0038,0.999958,0.989077,-0.001558,5008,0.145792,0.884086,True
9,10,QPU_IBM,da137vu3kjvs73868gp0,ibm_marrakesh,0,10000,5021,4979,0.5021,0.4979,0.0021,0.999987,0.993953,-0.003717,5019,0.361788,0.717510,True


In [ ]:
#Resumo descritivo dos 30 lotes
colunas_metricas = [
    "p1", "vies", "shannon", "min_entropia",
    "autocorr_lag1", "runs_z", "runs_p"
]

resumo = df_metricas[colunas_metricas].agg(["mean", "std", "median", "min", "max"]).T
q1 = df_metricas[colunas_metricas].quantile(0.25)
q3 = df_metricas[colunas_metricas].quantile(0.75)

resumo["q1"] = q1
resumo["q3"] = q3
resumo["iiq"] = q3 - q1

arquivo_resumo = PASTA_RESULTADOS / "qpu_resumo_descritivo.csv"
resumo.to_csv(arquivo_resumo)

print("Arquivo salvo em:", arquivo_resumo)
resumo


Arquivo salvo em: /content/results/qpu_resumo_descritivo.csv


,mean,std,median,min,max,q1,q3,iiq
p1,0.502993,0.005349,0.503100,0.493400,0.512500,0.498775,0.506925,0.008150
vies,0.005120,0.003280,0.003850,0.001100,0.012500,0.002400,0.006975,0.004575
shannon,0.999894,0.000125,0.999957,0.999549,0.999997,0.999859,0.999983,0.000124
min_entropia,0.985331,0.009346,0.988934,0.964376,0.996830,0.980014,0.993092,0.013078
autocorr_lag1,-0.002573,0.009697,-0.004011,-0.026762,0.015530,-0.008578,0.004723,0.013301
runs_z,0.247336,0.969618,0.391148,-1.562926,2.666063,-0.482291,0.847777,1.330069
runs_p,0.479817,0.265279,0.486712,0.007675,0.949265,0.258831,0.656449,0.397618


In [ ]:
nao_rejeita = int(df_metricas["runs_nao_rejeita_h0_alfa_005"].sum())
rejeita = len(df_metricas) - nao_rejeita
print(f"Runs Test — não rejeita H0 em {nao_rejeita}/30 lotes")
print(f"Runs Test — rejeita H0 em {rejeita}/30 lotes")
print("Rejeições individuais não implicam falha automática da fonte.")


Runs Test — não rejeita H0 em 29/30 lotes
Runs Test — rejeita H0 em 1/30 lotes
Rejeições individuais não implicam falha automática da fonte.


In [ ]:
#Condicionamento por von Neumann
def von_neumann(bits):
    saida = []
    for i in range(0, len(bits) - 1, 2):
        par = bits[i:i+2]
        if par == "01":
            saida.append("0")
        elif par == "10":
            saida.append("1")
    return "".join(saida)

lotes_vn = [von_neumann(bits) for bits in lotes]
total_vn = sum(len(x) for x in lotes_vn)
taxa_vn = total_vn / sum(len(x) for x in lotes)
print("Bits após von Neumann:", total_vn)
print(f"Taxa útil: {taxa_vn:.4%}")


Bits após von Neumann: 75188
Taxa útil: 25.0627%


In [ ]:
# Condicionamento por SHA-256
def bits_para_bytes(bits):
    return bytes(int(bits[i:i+8], 2) for i in range(0, len(bits), 8))

def sha256_condicionamento(bits, bloco_bits=1024):
    saida = []
    for inicio in range(0, len(bits) - bloco_bits + 1, bloco_bits):
        bloco = bits[inicio:inicio + bloco_bits]
        digest = hashlib.sha256(bits_para_bytes(bloco)).digest()
        saida.append("".join(f"{byte:08b}" for byte in digest))
    return "".join(saida)

lotes_sha = [sha256_condicionamento(bits) for bits in lotes]
total_sha = sum(len(x) for x in lotes_sha)
taxa_sha = total_sha / sum(len(x) for x in lotes)
print("Bits após SHA-256:", total_sha)
print(f"Taxa útil: {taxa_sha:.4%}")


Bits após SHA-256: 69120
Taxa útil: 23.0400%


In [ ]:
def dataframe_condicionado(nome, lotes_condicionados):
    linhas = []
    for i, bits in enumerate(lotes_condicionados, start=1):
        linha = {"lote": i, "condicao": nome}
        linha.update(metricas(bits))
        linhas.append(linha)
    return pd.DataFrame(linhas)

df_vn = dataframe_condicionado("von_neumann", lotes_vn)
df_sha = dataframe_condicionado("sha256", lotes_sha)

df_condicionamento = pd.concat([df_vn, df_sha], ignore_index=True)
arquivo_condicionamento = PASTA_RESULTADOS / "qpu_condicionamento_reanalise.csv"
df_condicionamento.to_csv(arquivo_condicionamento, index=False)

print("Arquivo salvo em:", arquivo_condicionamento)
df_condicionamento


Arquivo salvo em: /content/results/qpu_condicionamento_reanalise.csv


,lote,condicao,n_bits,n0,n1,p0,p1,vies,shannon,min_entropia,autocorr_lag1,numero_corridas,runs_z,runs_p,runs_nao_rejeita_h0_alfa_005
0,1,von_neumann,2512,1278,1234,0.508758,0.491242,0.008758,0.999779,0.974949,0.015243,1237,-0.783106,0.433565,True
1,2,von_neumann,2479,1260,1219,0.508269,0.491731,0.008269,0.999803,0.976335,-0.022058,1267,1.078612,0.280761,True
2,3,von_neumann,2580,1264,1316,0.489922,0.510078,0.010078,0.999707,0.971212,0.000741,1289,-0.058151,0.953628,True
3,4,von_neumann,2502,1222,1280,0.488409,0.511591,0.011591,0.999612,0.966938,-0.009721,1263,0.467048,0.640465,True
4,5,von_neumann,2495,1243,1252,0.498196,0.501804,0.001804,0.999991,0.994805,0.042487,1195,-2.141951,0.032197,False
5,6,von_neumann,2577,1284,1293,0.498254,0.501746,0.001746,0.999991,0.994970,-0.046594,1349,2.345277,0.019013,False
6,7,von_neumann,2519,1211,1308,0.480746,0.519254,0.019254,0.998930,0.945489,-0.006259,1266,0.294085,0.768693,True
7,8,von_neumann,2445,1222,1223,0.499796,0.500204,0.000204,1.000000,0.999410,-0.009002,1234,0.424793,0.670988,True
8,9,von_neumann,2510,1247,1263,0.496813,0.503187,0.003187,0.999971,0.990833,-0.013194,1272,0.640913,0.521579,True
9,10,von_neumann,2537,1270,1267,0.500591,0.499409,0.000591,0.999999,0.998295,-0.055206,1339,2.760272,0.005775,False


In [ ]:
def resumo_global(nome, sequencias):
    bits = "".join(sequencias)
    m = metricas(bits)
    return {
        "condicao": nome,
        "n_bits": m["n_bits"],
        "n0": m["n0"],
        "n1": m["n1"],
        "p0": m["p0"],
        "p1": m["p1"],
        "vies": m["vies"],
        "shannon": m["shannon"],
        "min_entropia": m["min_entropia"],
        "autocorr_lag1": m["autocorr_lag1"],
    }

df_global = pd.DataFrame([
    resumo_global("QPU_bruta", lotes),
    resumo_global("von_neumann", lotes_vn),
    resumo_global("sha256", lotes_sha),
])

df_global["taxa_util"] = df_global["n_bits"] / (30 * 10_000)
arquivo_global = PASTA_RESULTADOS / "qpu_resumo_global.csv"
df_global.to_csv(arquivo_global, index=False)

print("Arquivo salvo em:", arquivo_global)
df_global


Arquivo salvo em: /content/results/qpu_resumo_global.csv


,condicao,n_bits,n0,n1,p0,p1,vies,shannon,min_entropia,autocorr_lag1,taxa_util
0,QPU_bruta,300000,149102,150898,0.497007,0.502993,0.002993,0.999974,0.991389,-0.002426,1.000000
1,von_neumann,75188,37565,37623,0.499614,0.500386,0.000386,1.000000,0.998888,0.001316,0.250627
2,sha256,69120,34657,34463,0.501403,0.498597,0.001403,0.999994,0.995956,-0.004797,0.230400


## Arquivos produzidos

- `results/qpu_reanalise_metricas.csv`
- `results/qpu_resumo_descritivo.csv`
- `results/qpu_condicionamento_reanalise.csv`
- `results/qpu_resumo_global.csv`

Esses arquivos são derivados exclusivamente dos dados brutos preservados no repositório. Nenhuma nova execução em hardware quântico é submetida por este notebook.
